# 운영 환경 모니터링

* 에이전트는 확률적으로 작동함
* 모든 시나리오를 망라하는 테스트를 작성할 수 없음

-> 모니터링이 배포된 에이전트 인프라의 신경계 역할

* 모니터링은 문제 탐지와 더불어 학습과 반복을 가속하는 촘촘한 피드백 루프의 중추

## 모니터링: 학습의 출발점

* 에이전트 실패의 근본 원인을 이해하는 일은 선제적 유지보수와 시스템 적응성을 위해 필수
* 최고의 에이전트 시스템은 피드백을 통해 시간이 갈수록 개선

* 에이전트 실패는 미묘함
* 도구는 성공했지만 오류가 연쇄됨
    * 모니터링이 이를 신속히 드러내야 하므로 프로덕션 가시성은 필수

* 실패를 장애 하나가 아닌 테스트 케이스로 봐야 함
* 에이전트가 운영에서 실패할 대마다 그 시나리오는 캡처되어 회귀 테스트로 전환되어야 함

* 에이전트처럼 확률적 시스템을 모니터링할 때 핵심 과제는 진짜 실패와 기대 가능한 변동을 구분하는 일

* 효과적인 모니터링은 인프라 신호와 의미적 행동을 모두 포괄

* 계층적 피드백 루프를 구축해야 함
    * 컨텍스트와 함께 런타임 이벤트를 계측해 로키, 템포, 그라파나 같은 백엔드로 스트리밍
    * 외부 크리틱을 통해 할루시네이션 점수나 드리프트 지표 같은 평가 신호를 실시간으로 덧붙임

-> 모든 것은 프로덕션 서비스에 쓰는 동일한 가시성 파이프라인의 일부가 될 수 있으며 되어야 함

* 계측 세부사항으로 들어가기 전에 무엇을 관찰할지 정의하는 것이 유익함

## 모니터링 스펙

* 가시성은 지연시간, 가용성 같은 기존의 인프라 지표뿐 아니라 할루시네이션률, 도구 효과, 사용자 입력의 분포 이동 같은 의미적 신호도 포착해야 함

### 그라파나

* 구성 가능성이 높아 에이전트를 중심으로 맞춤형 가시성을 구축하려는 팀에 유연한 선택지

### ELK 스택

* Elasticsearch, Logstash/Fluentd, Kibana
* 강력한 검색과 분석에 중점을 둔 성숙한 옵션으로, 기존 엔터프라이즈 환경을 AI 워크로드로 확장하는데 자주 활용

### 어라이즈 피닉스

* LLM 트레이싱과 평가에 초점을 맞춘, 기존 환경에 에이전트 모니터링을 확장하는 디버그 지향 도구

### 시그노즈

* 지표, 트레이스, 로그를 한의 도구로 통합한 오픈텔레메트리 네이티브 플랫폼
* 기본 모니터링을 간결하게 확장하기에 적합함

### 랭퓨즈

* 파운데이션 모델과 에이전트 가시성에 특화되어 에이전트용 의미 중심 트레이싱을 기존 스택에 손쉽게 확장

##  프로젝트에 적합한 스택

* 먼저 현재 환경을 평가

* 모니터링과 관측 가능성 스택 비교

    | 스택 | 핵심 강점 | 적합한 대상 | 트레이드오프(그라파나 대비) |
    |-----|-----|-----|-----|
    | 그라파나<br>+ 로키/템포 | 조합 가능성과 시각화 | 엔터프라이즈 운영 | 관리해야 할 컴포넌트가 더 많음 |
    | ELK 스택 | 고급 검색/분석 | 대규모 로그 | 리소스 사용량 증가 |
    | 피닉스 | 트레이싱과 디버깅 | 개발 반복 | 프로덕션 확장 한계 |
    | 시그노즈 | 통합형이면서 경량 | 스타트업/ML 팀 | 확장성 낮음 |
    | 랭퓨즈 | 파운데이션 모델/에이전트 특화 평가 | 시맨틱 모니터링 | 인프라 커버리지 범위가 좁음 |

* 관측 가능성 생태계에는 확장성, 사용 편의성, LLM 특화 기능 등에서 각기 강점을 지닌 강력한 대안이 있음

## 오픈텔레메트리 계측

* 효과적인 모니터링 루프를 구축하는 첫 단계는 계측
* 오픈텔레메트리는 트레이스, 지표, 로그 전반에 걸친 구조화되고 상호운용 가능한 텔레메트리의 기반을 제공하며 랭그래프 기반 에이전트 시스템과도 잘 통합됨

* 랭그래프는 비동기 함수 호출의 그래프로 구성
    * 그래프의 각 노드는 에이전트 워크플로으 기능적 단계를 의미

* 각 노드마다 함수 시작 시 스팬을 시작하고 관련 메타데이터로 주석을 다는 것을 권장
* 이런 계측은 큰 아키텍처 변경으 요구하지 않음

* 랭그래프 노드를 트레이스 스팬으로 감싸는 단순화된 예시

In [ ]:
from opentelemetry import trace
tracer = trace.get_tracer("agent")

async def call_tool_node(context):
    with tracer.start_as_current_span("call_tool", attributes={
        "tool": context.tool_name,
        "input_tokens": context.token_usage.input,
        "output_tokens": context.token_usage.output
    }):
        result = await call_tool(context)
        return result

* 스팬에는 이벤트, 하위 스팬, 자동 에러 태깅을 위한 예외 캡처를 포함할 수 있음
* 이런 트레이스는 템포나 예거같은 백엔드로 실시간 내보내기 되며 그라파나에서 로그와 지표와 함께 시각화됨

* 오픈텔레메트리는 트레이스 외에도 구조화된 로그와 런타임 지표를 방출할 수 있음
    * 이런 지표는 장기 성능을 추적하고 초기 열화 신호를 감지하는 대시보드와 알림을 만드는 데 유용함

* 계측 범위는 신중해야 함
    * 정보가 지나치면 노이즈가 되고 부족하면 근본 원인 분석이 어려워짐
    * 핵심은 각 단계에 꼭 필요한 컨텍스트만 적절히 붙이는 것

* 템포는 트레이스 백엔드 역할
    * 랭그래프에서 계측한 모든 스팬, 즉 도구 호출, 계획 생성, 폴백은 분산 트레이스의 일부
    * 템포는 이러한 트레이스를 고도로 확장 가능한 방식으로 저장하고 심층 쿼리를 지원

* 로키는 로그 집계 레이어로 작동
    * 인프라 전반에서 발생하는 구조화된 로그를 수집
    * 각 랭그래프 노드는 실행 중에 구조화된 로그 이벤트를 내보낼 수 있음
    * 전문 검색, 역할 기반 보기, 더 높은 수집 처리량이 필요하다면 엘라스틱서치, 데이터독 로그, 허니콤 같은 상용 옵션도 고려할 수 있음

* 그라파나는 이 두 데이터 스트림을 단일 화면으로 통합
    * 로키의 로그와 템포의 트레이스를 나란히 탐색하는 시각화 레이어를 제공
    * 그라파나 안에서 실시간 트레이스 데이터를 표시하는 대시보드를 구성하고 개별 요청을 드릴다운하며 구조화된 로그를 성능 지표와 연관지을 수 있음
    * 사용자 정의 경보 규칙을 만들 수 있음

* 오픈텔레메트리, 템포, 로키, 그라파나는 에이전트 시스템을 위한 오픈 소스 관측 가능성 스택을 이룸
* 작동 과정을 깊이 있게 관찰하고 근본 원인 분석을 빠르게 수행하며 과거 추세를 평가하고 사전적 이상 탐지를 가능하게 함  

-> 이러한 통합은 원시 텔레메트리를 운영 인텔리전스로, 운영 인텔리전스를 개발 가속제로 전환

## 시각화와 알림

* 그라파나는 단순 대시보드 도구를 넘어 관측 가능성의 운영 프런트엔드로서 시그널을 스토리로, 지표를 실행으로 전환

* 그라파나는 로키와 템포를 네이티브 데이터 소스로 매끄럽게 연결
    * 성능이 저하되거나 엣지 케이스 버그가 발생할 때 다단계 에이전트 작동 과정을 디버깅하는 데 매우 유용함

* 로키의 경우 로키 플러그인으로 에이전트 실행 중 방출된 구조화 로그 이벤트를 질의할 수 있음

* 그라파나의 진정한 강점은 에이전트의 의미론과 성공 기준에 맞춘 맞춤형 대시보드를 구성하는 데 있음
* 각 패널은 시스템 성능을 시각화할 뿐 아니라 지속적인 개발을 이끎

* 그라파나는 커스텀 알림도 지원

* 알림은 대시보드를 실시간으로 보지 않더라도 팀이 회귀와 이상 징후를 즉시 인지하게 함
* 로키 로그와 템포 트레이스와 결합하면 피드백 루프를 빠르게 닫을 수 있음
* 그라파나의 알림 시스템은 확장성이 높아 페이저듀티 같은 인기 장애 관리 도구와 자연스럽게 통합되며 온콜 팀으로 알림을 전달해 할루시네이션 급증이나 작업 실패 같은 고심각도 이슈가 자동 호출과 확인을 포함한 구조화된 대응 워크플로를 트리거하도록 보장

* 그라파나를 에이전트 개발 라이프사이클에 깊이 통합하면 배포된 시스템을 위한 살아있는 인터페이스가 생김

## 모니터링 패턴

* 확률적이고 적응적이며 완전히 예측하기 어려운 에이전틱 시스템에 어떻게 배보팔 것인가?  
    -> 실험의 리스크를 줄이고 프로덕션 변경에 안전망을 두는 모니터링을 고려한 개발 패턴을 채택

### 섀도 모드

* 새롭거나 실험적인 에이전트 버전을 현재 프로덕션 에이전트와 나란히 실행
    * 동일한 입력을 처리, 출력은 사용자에게 제공 x

-> 실제 환경에서의 새 에이전트의 작동 과정을 로그와 트레이스로 관찰하되 사용자 경험에는 영향 x

* 섀도 모드는 더 안전한 혁신을 가능하게 함

### 카나리 배포

* 카나리 배포는 새 에이전트 버전을 실제 사용자 일부에게만 제공하고 나머지는 기준 버전과 상호작용하게 함

* 핵심은 그라파나 대시보드
    * 버전 태그로 모든 지표와 트레이스를 필터링해 카나리와 기준 에이전트를 직접 비교할 수 있음

* 작동이 양호하면 점진적으로 확대하고 그렇지 않으면 사용자 영향 최소화 상태에서 즉시 롤백 가능  
    -> 운영상의 안전성을 제공

### 희귀 트레이스 수집

* 에이전트가 프로덕션에서 실패할 때마다 학습 기회가 생김
* 이러한 실패 트레이스나 로그 스냅샷을 테스트 스위트로 자동 내보내면 지속적으로 갱신되는 회귀 코퍼스를 구축할 수 있음

* 프로덕션 실패를 학습 시그널로 전환하는 방식
* 실패한 도구 호출이나 불일치한 출력은 새로운 테스트 케이스가 됨
* 수정이 반영된 뒤 이 트레이스를 재실행하면 통과해야 함
* 시간이 지남에 따라 실제 엣지 케이스로 평가 집합이 강화되고 동일한 실패 양상의 재발을 방지함

### 자가 치유 에이전트

* 모니터링은 실패를 탐지하는 데 그치지 않고 에이전트가 회복하도록 도움
* 자체 텔레메트리를 실시간으로 읽도록 설계된 에이전트는 문제가 감지되면 폴백 메커니즘을 실행할 수 있음

* 자가 치유 작동 방식은 상세한 모니터링 데이터로 뒷받침될 때 가장 효과적

## 사용자 피드백

* 사용자 피드백은 보완적 관점, 즉 에이전트가 인간의 기대를 얼마나 충족하는지에 대한 직접 인사이트를 제공
* 피드백은 암시적이거나 명시적일 수 있음
    * 두 형태 모두 실시간 시그널을 제공하며 모니터링 스택에 통합해야 함

* 실무에서는 작업 포기율, 재질의 빈도 같은 암시적 피드백 지표를 로키에 로깅, 집계하고 다른 성능 지표와 동일하게 그라파나에서 시각화할 수 있음

* 무엇보다 사용자 피드백은 개선 루프를 구동

## 분포 변화

* 에이전트 기반 시스템을 모니터링할 때 분포 변화를 식별하고 관리하는 건 어렵지만 결정적으로 중요한 과제
    * 이는 시간이 지나며 에이전트 환경의 통계적 특성이 바뀔 때 발생함

* 작업 성공률, 도구 호출 실패, 토큰 사용량 추세나 할루시네이션 빈도 같은 시맨틱 지표를 추적하는 대시보드는 초기 신호를 드러낼 수 있음

* 복원력 있는 시스템을 구축하려면 이러한 변화에도 대응해야 함
    * 일시적 변화는 임계값 조정이나 파싱 로직 업데이트로 흡수하고 지속적 변화는 워크플로 재학습이나 새 API에의 적응이 필요함
    * 강력한 관측 가능성 스택이 제공하는 실시간 가시성으로 드리프트가 실패로 번지기 전에 선제적으로 대응할 수 있음

## 지표 소유권과 기능 간 거버넌스

* 팀이 에이전틱 시스템을 배포하면 누가 어떤 지표를 소유하는지가?
    * 기존 소프트웨어 스택에서는 구분이 뚜렷함
        * 인프라: 지연시간, 가용성
        * 프로덕트: 전환이나 사용자 성공
        * 머신러닝: 모델을 구축하고 그 건강과 성능을 관리하며 엔지니어링과 프로덕트 영향 모두에 책임
    * 다만 파운데이션 모델이 구동하는 에이전트는 이런 경계를 따르지 않음  
        -> 모니터링 전략도 그래야 함

* 모니터링 지표와 기능 간 책임의 RACI 매트릭스

| 지표/활동 | 프로덕트 팀 | ML 엔지니어 | 인프라/SRE 팀 |
|-----|-----|-----|-----|
| 지연시간 | A(사용자 영향 소유) /<br>C(UX 임계값 자문) | R(프롬프트/모델 최적화)<br>/ I(회귀 통보) | R(인프라 원인 모니터링) /<br>C(스케일링 자문) |
| 할루시네이션 비율 | C(사용자 피드백 컨텍스트 제공) / <br>I(추세 통보) | A/R(평가를 통한 탐지/완화 소유) | I(알림 설정을 위한 통보) |
| 작업 성공률 | A(제품 목표 소유) /<br>R(성공 기준 정의) | C(모델 개선 자문) | I(시스템 신뢰성과의 연계 통보) |
| 토큰 사용량/비용 | C(비즈니스 영향 자문) | R(생성 최적화) /<br>I(급증 통보) | A(예산/스케일링 소유) /<br>R(인프라 효율 모니터링) |
| 분포 변화 | I(프로덕트 조정 통보) | A/R(임베딩/평가로 탐지) | C(데이터 파이프라인 안정성 자문) |
| 폴백/재시도 빈도 | C(UX 폴백 자문) | R(계획 로직 개선) | A(신뢰성 소유) /<br>I(패턴 통보) |
| 사용자 피드백/감성 | A/R(집계와 우선순위 소유) | C(모델 연계 자문) | I(운영 알림 통보) |
| 대시보드 유지보수와 트리아지 의식 | C(프로덕트 컨텍스트 제공) | C(ML 인사이트 제공) | A/R(플랫폼 소유, 기능간 리뷰 주관) |

* 도구가 루프 안에서 네 번 호출되고 이어 긴 생성, 모호한 응답, 사용자 이탈로 끝나는 트레이스는 단지 엔지니어링 세부 사항이 아님  
    -> 프로덕트의 실패
* 이런 현상은 로키와 템포 같은 공유 플랫폼으로 로그와 스팬을 라우팅할 때만 보임

* 이를 작동시키는 실천법
    * 버전 태그와 시맨틱 지표를 갖춘 공유 관측 가능성 대시보드 사용
    * 스팬과 로그에 제품 컨텍스트를 태깅
    * 출시나 큰 회귀 이후 프로덕트, 인프라, ML이 함께 텔레메트리를 리뷰하는 기능 간 트리아지 의식 생성
    * 이중잣대 피하기

* 모니터링 스택은 단순히 장애를 감지하는 도구가 아닌 엔지니어링, ML, 프로덕트가 시스템이 무엇을 하고 있는지 얼마나 잘 수행하는지 어디를 개선해야 하는지에 대해 같은 언어로 대화하도록 해주는 인터페이스